# 特徵值與特徵向量 (Eigenvalues and Eigenvectors)

本筆記本介紹線性代數中最重要的概念之一——**特徵值與特徵向量**：

1. **定義** — $A\mathbf{v}=\lambda\mathbf{v}$ 是什麼意思？
2. **幾何意義** — 矩陣如何「拉伸／翻轉」特定方向
3. **特徵多項式** — $\det(A-\lambda I)=0$
4. **逐步手算** — 用 SymPy 跟一次完整流程
5. **`eigenvals` / `eigenvects`** — SymPy 快捷方法
6. **特徵空間與重數** — 代數重數 vs 幾何重數
7. **對角化** — $A=PDP^{-1}$
8. **對稱矩陣** — 實對稱 ⇒ 正交對角化
9. **應用** — 用對角化算 $A^n$
10. **綜合練習**

In [1]:
import sympy as sp
from IPython.display import display, Math


def display_large(expr):
    """Display a sympy expression in LaTeX with large font size."""
    display(Math(r"\large{%s}" % sp.latex(expr)))


def display_huge(expr):
    """Display a sympy expression in LaTeX with huge font size."""
    display(Math(r"\huge{%s}" % sp.latex(expr)))

## 1. 什麼是特徵值與特徵向量？

給定方陣 $A$（$n\times n$），若存在**非零**向量 $\mathbf{v}$ 與純量 $\lambda$，使得

$$
A\mathbf{v} = \lambda\mathbf{v}
$$

則稱：

| 名稱 | 英文 | 符號 |
|------|------|------|
| **特徵值** | eigenvalue | $\lambda$ |
| **特徵向量** | eigenvector | $\mathbf{v}$（$\mathbf{v}\neq\mathbf{0}$） |

直觀意義：對特徵向量做線性變換 $A$，結果只是**沿同一方向伸縮**（乘上 $\lambda$），**方向不變**（若 $\lambda<0$ 則反向）。

等價改寫：

$$
(A-\lambda I)\mathbf{v} = \mathbf{0}
$$

也就是：非零 $\mathbf{v}$ 落在 $A-\lambda I$ 的**零空間**裡。因此 $\lambda$ 是特徵值 $\Leftrightarrow$ $A-\lambda I$ **奇異** $\Leftrightarrow$

$$
\det(A-\lambda I) = 0
$$

In [2]:
# Quick check: Av = λv for a known pair
A = sp.Matrix([[2, 1], [1, 2]])
v = sp.Matrix([1, 1])
lam = 3

print("A =")
display_huge(A)
print("v =")
display_huge(v)
print("A v =")
display_huge(A * v)
print("λ v =")
display_huge(lam * v)
print("Av == λv?", A * v == lam * v)

A =


<IPython.core.display.Math object>

v =


<IPython.core.display.Math object>

A v =


<IPython.core.display.Math object>

λ v =


<IPython.core.display.Math object>

Av == λv? True


## 2. 幾何意義

對同一個矩陣 $A$，不同方向的向量被「拉」的程度不同：

- 特徵方向：$A\mathbf{v}$ 與 $\mathbf{v}$ **平行**
- 一般方向：$A\mathbf{w}$ 會**旋轉＋伸縮**，不再平行於 $\mathbf{w}$

$\lambda$ 告訴你伸縮倍率：

| $\lambda$ | 效果 |
|-----------|------|
| $\lambda>1$ | 沿該方向拉長 |
| $0<\lambda<1$ | 沿該方向縮短 |
| $\lambda=1$ | 該方向不動 |
| $\lambda=0$ | 壓成零向量（落在零空間） |
| $\lambda<0$ | 反向再伸縮 |

In [3]:
A = sp.Matrix([[2, 1], [1, 2]])

# Eigen-direction: stays parallel
v_eigen = sp.Matrix([1, 1])
# Generic direction: gets rotated
w = sp.Matrix([1, 0])

Av = A * v_eigen
Aw = A * w

print("Eigen direction v = [1, 1]^T")
print("A v =")
display_huge(Av)
print("Parallel to v?", Av[0] * v_eigen[1] == Av[1] * v_eigen[0])

print("\nGeneric direction w = [1, 0]^T")
print("A w =")
display_huge(Aw)
print("Parallel to w?", Aw[0] * w[1] == Aw[1] * w[0])

Eigen direction v = [1, 1]^T
A v =


<IPython.core.display.Math object>

Parallel to v? True

Generic direction w = [1, 0]^T
A w =


<IPython.core.display.Math object>

Parallel to w? False


## 3. 特徵多項式

定義**特徵多項式（characteristic polynomial）**為

$$
p_A(\lambda) = \det(A-\lambda I)
$$

（有些書寫成 $\det(\lambda I-A)$，只差一個符號 $(-1)^n$。）

特徵值就是 $p_A(\lambda)=0$ 的根。對 $n\times n$ 矩陣，$p_A$ 是 $n$ 次多項式，在複數體上恰有 $n$ 個根（計重數）。

SymPy：

- `A.charpoly(lam)` — 特徵多項式
- `A.eigenvals()` — 特徵值（字典：值 → 代數重數）
- `A.eigenvects()` — `(λ, 重數, [基底向量…])` 列表

In [4]:
lam = sp.symbols("lambda")
A = sp.Matrix([[2, 1], [1, 2]])

print("A - λI =")
display_huge(A - lam * sp.eye(2))

p = A.charpoly(lam)
print("Characteristic polynomial:")
display_large(p.as_expr())

print("Roots (eigenvalues):")
display_large(sp.factor(p.as_expr()))
print("eigenvals() =")
display_large(A.eigenvals())

A - λI =


<IPython.core.display.Math object>

Characteristic polynomial:


<IPython.core.display.Math object>

Roots (eigenvalues):


<IPython.core.display.Math object>

eigenvals() =


<IPython.core.display.Math object>

## 4. 逐步手算：從定義到特徵向量

標準流程（以 $2\times 2$ 為例）：

1. 算 $A-\lambda I$
2. 令 $\det(A-\lambda I)=0$，解出 $\lambda$
3. 對每個 $\lambda$，解 $(A-\lambda I)\mathbf{v}=\mathbf{0}$（找零空間）
4. 得到特徵空間 $E_\lambda = N(A-\lambda I)$

In [5]:
lam = sp.symbols("lambda")
A = sp.Matrix([[4, 2], [1, 3]])

print("Step 0: A =")
display_huge(A)

# Step 1–2: characteristic equation
M = A - lam * sp.eye(2)
print("Step 1: A - λI =")
display_huge(M)

char_eq = sp.Eq(M.det(), 0)
print("Step 2: det(A - λI) = 0")
display_large(char_eq)
display_large(sp.Eq(sp.expand(M.det()), 0))

eigs = sp.solve(M.det(), lam)
print("Eigenvalues:")
display_large(eigs)

Step 0: A =


<IPython.core.display.Math object>

Step 1: A - λI =


<IPython.core.display.Math object>

Step 2: det(A - λI) = 0


<IPython.core.display.Math object>

<IPython.core.display.Math object>

Eigenvalues:


<IPython.core.display.Math object>

In [6]:
# Step 3–4: nullspace for each eigenvalue
for eig in eigs:
    print(f"\n--- λ = {eig} ---")
    N = A - eig * sp.eye(2)
    print("A - λI =")
    display_huge(N)
    print("RREF:")
    display_huge(N.rref()[0])
    basis = N.nullspace()
    print("Nullspace basis (eigenvector):")
    for v in basis:
        display_huge(v)
        print("Check A v = λ v:")
        display_large(sp.Matrix.hstack(A * v, eig * v))


--- λ = 2 ---
A - λI =


<IPython.core.display.Math object>

RREF:


<IPython.core.display.Math object>

Nullspace basis (eigenvector):


...

<IPython.core.display.Math object>

Nullspace basis (eigenvector):


<IPython.core.display.Math object>

Check A v = λ v:


<IPython.core.display.Math object>

## 5. SymPy 快捷：`eigenvals` 與 `eigenvects`

`A.eigenvects()` 回傳列表，每個元素是

$$
(\lambda,\;\text{代數重數},\;[\mathbf{v}_1,\mathbf{v}_2,\ldots])
$$

其中向量列表是 $E_\lambda$ 的一組基底（幾何重數 = 列表長度）。

In [7]:
A = sp.Matrix([[4, 2], [1, 3]])

print("eigenvals():")
display_large(A.eigenvals())

print("\neigenvects():")
for val, mult, vecs in A.eigenvects():
    print(f"λ = {val}, algebraic multiplicity = {mult}")
    print("eigenvectors:")
    for v in vecs:
        display_huge(v)

eigenvals():


<IPython.core.display.Math object>


eigenvects():
λ = 2, algebraic multiplicity = 1
eigenvectors:


<IPython.core.display.Math object>

λ = 5, algebraic multiplicity = 1
eigenvectors:


<IPython.core.display.Math object>

## 6. 特徵空間與重數

對特徵值 $\lambda$：

| 概念 | 定義 |
|------|------|
| **特徵空間** $E_\lambda$ | $N(A-\lambda I)=\{\mathbf{v}:A\mathbf{v}=\lambda\mathbf{v}\}$（含 $\mathbf{0}$） |
| **代數重數** | 特徵多項式中 $(\lambda-\lambda_0)$ 的次數 |
| **幾何重數** | $\dim E_\lambda$（獨立特徵向量個數） |

永遠有

$$
1 \le \text{幾何重數} \le \text{代數重數}
$$

若某個 $\lambda$ 的幾何重數 **小於** 代數重數，矩陣**不可對角化**（defectiveness）。

In [8]:
# Repeated eigenvalue with full geometric multiplicity
A_good = sp.Matrix([[2, 0, 0], [0, 2, 0], [0, 0, 3]])
print("A (diagonalizable):")
display_huge(A_good)
print("eigenvals:")
display_large(A_good.eigenvals())
for val, alg, vecs in A_good.eigenvects():
    print(f"λ={val}: alg={alg}, geo={len(vecs)}")

# Defective: Jordan block — only one independent eigenvector for λ=2 (alg=2)
A_bad = sp.Matrix([[2, 1], [0, 2]])
print("\nA (defective, not diagonalizable):")
display_huge(A_bad)
print("Characteristic polynomial:")
display_large(A_bad.charpoly().as_expr())
for val, alg, vecs in A_bad.eigenvects():
    print(f"λ={val}: alg={alg}, geo={len(vecs)}")
    for v in vecs:
        display_huge(v)

A (diagonalizable):


<IPython.core.display.Math object>

eigenvals:


<IPython.core.display.Math object>

λ=2: alg=2, geo=2
λ=3: alg=1, geo=1

A (defective, not diagonalizable):


<IPython.core.display.Math object>

Characteristic polynomial:


<IPython.core.display.Math object>

λ=2: alg=2, geo=1


<IPython.core.display.Math object>

## 7. 對角化 $A = PDP^{-1}$

若 $A$ 有 $n$ 個**線性獨立**的特徵向量 $\mathbf{v}_1,\ldots,\mathbf{v}_n$（對應 $\lambda_1,\ldots,\lambda_n$），令

$$
P = [\mathbf{v}_1\;\cdots\;\mathbf{v}_n],\qquad
D = \operatorname{diag}(\lambda_1,\ldots,\lambda_n)
$$

則

$$
AP = PD \quad\Rightarrow\quad A = PDP^{-1}
$$

判別：$A$ 可對角化 $\Leftrightarrow$ 每個特徵值的幾何重數 = 代數重數 $\Leftrightarrow$ 存在完整的特徵向量基底。

SymPy：`A.diagonalize()` 回傳 `(P, D)`。

In [9]:
A = sp.Matrix([[4, 2], [1, 3]])

P, D = A.diagonalize()
print("P (columns = eigenvectors):")
display_huge(P)
print("D (eigenvalues on diagonal):")
display_huge(D)

print("Check P^{-1} A P = D:")
display_huge(sp.simplify(P.inv() * A * P))

print("Check A = P D P^{-1}:")
display_huge(sp.simplify(P * D * P.inv()))
print("Match original A?", sp.simplify(P * D * P.inv()) == A)

P (columns = eigenvectors):


<IPython.core.display.Math object>

D (eigenvalues on diagonal):


<IPython.core.display.Math object>

Check P^{-1} A P = D:


<IPython.core.display.Math object>

Check A = P D P^{-1}:


<IPython.core.display.Math object>

Match original A? True


In [10]:
# Defective matrix cannot be diagonalized
A_bad = sp.Matrix([[2, 1], [0, 2]])
try:
    A_bad.diagonalize()
except Exception as e:
    print("diagonalize() failed as expected:")
    print(type(e).__name__ + ":", e)

diagonalize() failed as expected:
MatrixError: Matrix is not diagonalizable


## 8. 對稱矩陣：正交對角化

**實對稱**矩陣（$A^{\mathsf{T}}=A$）有漂亮性質：

1. 所有特徵值都是**實數**
2. 不同特徵值的特徵向量**彼此正交**
3. 一定可以**正交對角化**：$A=QDQ^{\mathsf{T}}$，其中 $Q^{\mathsf{T}}Q=I$

這把第 6 章的正交概念與特徵值連在一起。

In [11]:
A = sp.Matrix([[2, 1], [1, 2]])
print("A (symmetric):")
display_huge(A)
print("A^T = A?", A.T == A)

print("Eigenvalues (all real):")
display_large(A.eigenvals())

# Collect eigenvectors and check orthogonality
vecs = []
for val, mult, basis in A.eigenvects():
    for v in basis:
        vecs.append(v)
        print(f"eigenvector for λ={val}:")
        display_huge(v)

print("v1 · v2 =")
display_large(vecs[0].dot(vecs[1]))

# Orthogonal diagonalization via normalized eigenvectors
Q, D = A.diagonalize(normalize=True)
print("Q (orthonormal columns):")
display_huge(sp.simplify(Q))
print("Q^T Q =")
display_huge(sp.simplify(Q.T * Q))
print("A = Q D Q^T?")
display_huge(sp.simplify(Q * D * Q.T))
print("Match?", sp.simplify(Q * D * Q.T) == A)

A (symmetric):


<IPython.core.display.Math object>

A^T = A? True
Eigenvalues (all real):


<IPython.core.display.Math object>

eigenvector for λ=1:


...

Q^T Q =


<IPython.core.display.Math object>

A = Q D Q^T?


<IPython.core.display.Math object>

Match? True


## 9. 應用：用對角化算 $A^n$

若 $A=PDP^{-1}$，則

$$
A^n = PD^n P^{-1}
$$

而 $D^n$ 只是把對角元各自取 $n$ 次方——比直接乘矩陣 $n$ 次省事得多，也適合求閉式。

In [12]:
A = sp.Matrix([[4, 2], [1, 3]])
P, D = A.diagonalize()
n = sp.symbols("n", integer=True, nonnegative=True)

D_n = D.applyfunc(lambda x: x**n)
A_n = sp.simplify(P * D_n * P.inv())

print("D =")
display_huge(D)
print("D^n =")
display_huge(D_n)
print("A^n = P D^n P^{-1} =")
display_huge(A_n)

# Verify for a concrete n
print("Check n=5: formula vs direct power")
display_huge(A_n.subs(n, 5))
display_huge(A**5)
print("Match?", A_n.subs(n, 5) == A**5)

D =


<IPython.core.display.Math object>

D^n =


<IPython.core.display.Math object>

A^n = P D^n P^{-1} =


<IPython.core.display.Math object>

Check n=5: formula vs direct power


<IPython.core.display.Math object>

<IPython.core.display.Math object>

Match? True


## 10. 綜合練習

考慮

$$
A = \begin{bmatrix} 1 & 2 & 0 \\ 2 & 1 & 0 \\ 0 & 0 & 3 \end{bmatrix}
$$

1. 確認 $A$ 是否對稱
2. 求所有特徵值與對應特徵向量
3. 對每個特徵值，比較代數重數與幾何重數
4. 對角化：$A=PDP^{-1}$（或正交對角化 $QDQ^{\mathsf{T}}$）
5. 用對角化算出 $A^3$，並與直接計算比對

In [13]:
A = sp.Matrix([[1, 2, 0], [2, 1, 0], [0, 0, 3]])

# 1. Symmetric?
print("1. A =")
display_huge(A)
print("Symmetric?", A.T == A)

# 2–3. Eigenpairs and multiplicities
print("\n2–3. Eigenvalues / multiplicities:")
for val, alg, vecs in A.eigenvects():
    print(f"λ = {val}, alg = {alg}, geo = {len(vecs)}")
    for v in vecs:
        display_huge(v)

# 4. Orthogonal diagonalization (symmetric)
Q, D = A.diagonalize(normalize=True)
print("\n4. Q =")
display_huge(sp.simplify(Q))
print("D =")
display_huge(D)
print("Q^T Q =")
display_huge(sp.simplify(Q.T * Q))
print("A = Q D Q^T?", sp.simplify(Q * D * Q.T) == A)

# 5. A^3 via diagonalization
A3_via = sp.simplify(Q * (D**3) * Q.T)
print("\n5. A^3 via Q D^3 Q^T =")
display_huge(A3_via)
print("Direct A**3 =")
display_huge(A**3)
print("Match?", A3_via == A**3)

1. A =


<IPython.core.display.Math object>

Symmetric? True

2–3. Eigenvalues / multiplicities:
λ = -1, alg = 1, geo = 1


<IPython.core.display.Math object>

λ = 3, alg = 2, geo = 2


...

A = Q D Q^T? True

5. A^3 via Q D^3 Q^T =


<IPython.core.display.Math object>

Direct A**3 =


<IPython.core.display.Math object>

Match? True


## 小結

| 概念 | SymPy / 公式 | 重點 |
|------|--------------|------|
| 定義 | $A\mathbf{v}=\lambda\mathbf{v}$ | $\mathbf{v}\neq\mathbf{0}$，方向不變 |
| 特徵方程 | `det(A-λI)=0` / `A.charpoly()` | 根 = 特徵值 |
| 特徵向量 | `(A-λI).nullspace()` | $E_\lambda=N(A-\lambda I)$ |
| 快捷 | `A.eigenvals()` / `A.eigenvects()` | 一次取出值與向量 |
| 重數 | 代數 ≥ 幾何 ≥ 1 | 幾何 < 代數 ⇒ 不可對角化 |
| 對角化 | `A.diagonalize()` → $PDP^{-1}$ | 需 $n$ 個獨立特徵向量 |
| 對稱矩陣 | `diagonalize(normalize=True)` | $A=QDQ^{\mathsf{T}}$，特徵向量可正交 |
| 矩陣次方 | $A^n=PD^nP^{-1}$ | 對角元各自乘方 |

**記住：**

- 特徵值／特徵向量描述「矩陣只做伸縮、不旋轉」的特殊方向
- 找 $\lambda$：解 $\det(A-\lambda I)=0$；找 $\mathbf{v}$：解零空間
- 可對角化 $\Leftrightarrow$ 有完整特徵向量基底
- 實對稱矩陣一定可以正交對角化